# ⚡ Actividad 00b — Colab como servidor vLLM con GPU
## Alto rendimiento · API OpenAI nativa · HuggingFace models · Ngrok

---

**vLLM** es un motor de inferencia optimizado para LLMs que expone una API 100% compatible con OpenAI.
Es significativamente más rápido que Ollama en throughput (tokens/segundo) gracias a técnicas como
PagedAttention y continuous batching.

```
Tu laptop / otra Colab
        │
        │  HTTP POST  (OpenAI format)
        ▼
  Ngrok Tunnel  ──►  Colab T4 GPU
                           │
                      vLLM Server
                      localhost:8000
                           │
                    Qwen2-0.5B-Instruct
                    (HuggingFace)
```

| | Ollama (00a) | vLLM (00b) |
|---|---|---|
| Setup | Simple | Más complejo |
| Velocidad | Buena | Excelente |
| API | Propia + OpenAI | OpenAI nativa |
| Modelos | Catálogo Ollama | HuggingFace directo |
| Memoria T4 | Eficiente | Más exigente |

**⏱ Duración:** 40–50 minutos | **🎯 Resultado:** API OpenAI-compatible sobre tu GPU en la nube

---
## PARTE 1 · Verificar GPU y configurar HuggingFace
**⏱ 3 minutos**

vLLM descarga modelos directamente desde HuggingFace. Para modelos con licencia (Llama, Mistral)
necesitas un token HF. Usamos `Qwen2-0.5B-Instruct` que es de acceso libre.

Guarda tu token HF en Colab Secrets como `HF_TOKEN` antes de continuar.

In [ ]:
import subprocess

# Verificar GPU: vLLM requiere CUDA para funcionar
resultado = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if resultado.returncode == 0:
    print(resultado.stdout)
    print('✅ GPU detectada — vLLM puede arrancar')
else:
    print('❌ vLLM REQUIERE GPU. Ve a Entorno de ejecución → Cambiar tipo → T4 GPU')
    raise SystemExit('GPU no disponible')

In [ ]:
from google.colab import userdata
import os

# Leer token de HuggingFace desde Colab Secrets
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    # Exportar como variable de entorno para que vLLM lo use automáticamente
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
    print('✅ Token HuggingFace configurado desde Colab Secrets')
except Exception:
    print('⚠️  No se encontró HF_TOKEN en Secrets.')
    print('   Para modelos abiertos (Qwen, TinyLlama) no es necesario.')
    print('   Para Llama/Mistral sí se requiere. Configúralo en Colab Secrets.')

---
## PARTE 2 · Instalar vLLM
**⏱ 5 minutos**

vLLM se instala con pip. La instalación incluye CUDA kernels optimizados
para máximo rendimiento en la GPU. Tarda unos minutos la primera vez.

In [ ]:
# Instalar vLLM y dependencias de cliente
# --quiet suprime la salida verbose de pip
!pip install -q vllm pyngrok openai requests
print('✅ vLLM y dependencias instaladas')

In [ ]:
# Verificar que vLLM se instaló correctamente
import vllm
print(f'vLLM versión: {vllm.__version__}')
print('✅ vLLM importado correctamente')

---
## PARTE 3 · Arrancar el servidor vLLM
**⏱ 5–8 minutos**

vLLM se lanza como un proceso servidor usando su módulo de API OpenAI.
Al arrancar, descarga el modelo de HuggingFace y lo carga en la GPU.

### ¿Cómo funciona PagedAttention?
vLLM usa **PagedAttention**: en lugar de reservar memoria contigua para el KV-cache
(como los transformers estándar), divide la memoria en páginas no contiguas.
Esto permite servir muchas peticiones simultáneas sin desperdiciar VRAM.

In [ ]:
import subprocess
import time
import requests

# 🔧 PARÁMETRO: modelo a servir (debe caber en ~12 GB VRAM de T4)
# Opciones probadas en T4:
#   'Qwen/Qwen2-0.5B-Instruct'       → 0.5B params, ~1 GB VRAM, muy rápido
#   'TinyLlama/TinyLlama-1.1B-Chat-v1.0' → 1.1B params, ~2.2 GB VRAM
#   'microsoft/phi-2'                  → 2.7B params, ~5.5 GB VRAM
MODELO_HF = 'Qwen/Qwen2-0.5B-Instruct'

# 🔧 PARÁMETRO: fracción de VRAM que vLLM puede usar (0.0 - 1.0)
# Valores altos = más throughput pero más riesgo de OOM
GPU_MEMORY_FRACTION = 0.85

print(f'Arrancando vLLM con modelo: {MODELO_HF}')
print(f'GPU memory utilization    : {GPU_MEMORY_FRACTION}')
print('(Descargando modelo de HuggingFace si no está en caché...)')

# Lanzar vLLM como proceso servidor en background
vllm_process = subprocess.Popen(
    [
        'python', '-m', 'vllm.entrypoints.openai.api_server',
        '--model', MODELO_HF,           # Modelo a cargar
        '--port', '8000',               # Puerto del servidor
        '--host', '0.0.0.0',            # Escuchar en todas las interfaces
        '--gpu-memory-utilization', str(GPU_MEMORY_FRACTION),
        '--max-model-len', '2048',      # Longitud máxima de contexto (reducir si OOM)
        '--dtype', 'float16',           # float16 para ahorrar VRAM en T4
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,           # Redirigir stderr a stdout para capturarlo
    text=True
)

print('Esperando que el servidor arranque (puede tardar 2-5 min)...', end='')

# Esperar hasta que el servidor responda en el puerto 8000
for intento in range(120):             # Esperar hasta 2 minutos
    try:
        resp = requests.get('http://localhost:8000/health', timeout=2)
        if resp.status_code == 200:
            print(f' listo en ~{intento}s')
            break
    except requests.exceptions.ConnectionError:
        print('.', end='', flush=True)
        time.sleep(1)
else:
    print('\n❌ El servidor no arrancó a tiempo.')
    print('   Revisa si el modelo cabe en la VRAM. Prueba con Qwen2-0.5B-Instruct.')

print('✅ Servidor vLLM activo en localhost:8000')

In [ ]:
# Consultar qué modelos tiene cargados el servidor
resp_models = requests.get('http://localhost:8000/v1/models')
modelos = resp_models.json()
print('=== MODELOS CARGADOS EN VLLM ===')
for m in modelos.get('data', []):
    print(f"  ID: {m['id']}")
print('✅ Servidor respondiendo correctamente')

---
## PARTE 4 · Probar el servidor localmente
**⏱ 5 minutos**

La API de vLLM es 100% compatible con OpenAI — mismos endpoints, mismos parámetros.
Cualquier código que usas con `openai.ChatCompletion` funciona sin cambios apuntando a vLLM.

In [ ]:
from openai import OpenAI

# Conectar el SDK de OpenAI al servidor vLLM local
client_local = OpenAI(
    base_url='http://localhost:8000/v1',   # vLLM en local
    api_key='vllm'                          # vLLM no valida el key, pero el SDK lo requiere
)

print('=== TEST: Chat Completion ===')
resp = client_local.chat.completions.create(
    model=MODELO_HF,             # Nombre exacto del modelo cargado
    messages=[
        {'role': 'system', 'content': 'Eres un asistente conciso que responde en español.'},
        {'role': 'user',   'content': '¿Qué es vLLM y en qué se diferencia de Ollama?'}
    ],
    max_tokens=200,              # Límite de tokens en la respuesta
    temperature=0.7              # Temperatura: 0=determinista, 1=creativo
)

print(f'Respuesta: {resp.choices[0].message.content}')
print(f'\nTokens usados: {resp.usage.prompt_tokens} prompt + {resp.usage.completion_tokens} completion')
print('✅ API OpenAI nativa funciona con vLLM')

In [ ]:
# vLLM soporta streaming nativo: el texto llega token a token
print('=== TEST: Streaming (tokens en tiempo real) ===')
print('Respuesta: ', end='', flush=True)

stream = client_local.chat.completions.create(
    model=MODELO_HF,
    messages=[{'role': 'user', 'content': 'Cuenta del 1 al 5 lentamente.'}],
    stream=True,                 # stream=True activa el modo streaming
    max_tokens=100
)

# Cada chunk llega tan pronto como el modelo genera ese token
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:                    # El último chunk viene con content=None
        print(delta, end='', flush=True)

print('\n✅ Streaming funciona correctamente')

---
## PARTE 5 · Exponer el servidor vLLM con Ngrok
**⏱ 5 minutos**

Igual que con Ollama, usamos Ngrok para crear un túnel público.
La diferencia: vLLM escucha en el puerto **8000** en lugar del 11434.

> ⚠️ **Seguridad:** la API de vLLM no tiene autenticación por defecto.
> Cualquiera con la URL puede usarla. No la compartas públicamente.

In [ ]:
from pyngrok import ngrok
from google.colab import userdata

# Leer token de Ngrok desde Colab Secrets
try:
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')
    print('✅ Token leído desde Colab Secrets')
except Exception:
    NGROK_TOKEN = 'PEGA_TU_TOKEN_AQUÍ'
    print('⚠️  Token manual. Configura Colab Secrets para mayor seguridad.')

ngrok.set_auth_token(NGROK_TOKEN)

# Cerrar túneles previos
ngrok.kill()

# Crear túnel apuntando al puerto 8000 de vLLM
tunnel = ngrok.connect(8000, 'http')
URL_PUBLICA = tunnel.public_url

print('=' * 60)
print(f'⚡ SERVIDOR vLLM DISPONIBLE EN:')
print(f'   {URL_PUBLICA}')
print('=' * 60)
print(f'\nEndpoints disponibles:')
print(f'  Chat completions : {URL_PUBLICA}/v1/chat/completions')
print(f'  Completions      : {URL_PUBLICA}/v1/completions')
print(f'  Modelos          : {URL_PUBLICA}/v1/models')
print(f'  Health           : {URL_PUBLICA}/health')
print(f'\nModelo activo: {MODELO_HF}')

---
## PARTE 6 · Conectarse desde clientes externos
**⏱ 8 minutos**

Al ser 100% compatible con OpenAI, cualquier librería que soporte `base_url`
puede apuntar al servidor vLLM sin cambiar el código.

In [ ]:
# === CLIENTE 1: OpenAI SDK desde URL pública ===
print('=== CLIENTE OpenAI SDK → vLLM via Ngrok ===')

client_ext = OpenAI(
    base_url=f'{URL_PUBLICA}/v1',
    api_key='vllm',
    default_headers={'ngrok-skip-browser-warning': 'true'}
)

resp_ext = client_ext.chat.completions.create(
    model=MODELO_HF,
    messages=[{'role': 'user', 'content': '¿Qué es el aprendizaje profundo? En dos oraciones.'}],
    max_tokens=150
)
print(f'Respuesta: {resp_ext.choices[0].message.content}')
print('✅ Cliente OpenAI SDK funciona via Ngrok')

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# === CLIENTE 2: LangChain con ChatOpenAI apuntando a vLLM ===
# ChatOpenAI acepta base_url para redirigir a cualquier servidor compatible
print('=== CLIENTE LangChain ChatOpenAI → vLLM ===')

llm_vllm = ChatOpenAI(
    model=MODELO_HF,
    openai_api_base=f'{URL_PUBLICA}/v1',   # URL del servidor vLLM
    openai_api_key='vllm',                  # Key ficticia (vLLM no la valida)
    max_tokens=200,
    model_kwargs={'headers': {'ngrok-skip-browser-warning': 'true'}}
)

mensajes = [
    SystemMessage(content='Eres un experto en IA que explica conceptos de forma simple.'),
    HumanMessage(content='¿Cuándo usarías vLLM en producción?')
]
respuesta_lc = llm_vllm.invoke(mensajes)
print(f'Respuesta: {respuesta_lc.content}')
print('✅ LangChain conectado al servidor vLLM')

In [ ]:
# === CLIENTE 3: curl desde terminal ===
print('Para usar desde tu terminal:')
print()
print(f'curl -s {URL_PUBLICA}/v1/chat/completions \\')
print('  -H "Content-Type: application/json" \\')
print('  -H "ngrok-skip-browser-warning: true" \\')
print('  -d \'{"model": "' + MODELO_HF + '", "messages": [{"role": "user", "content": "Hola"}], "max_tokens": 100}\'')
print()
print('# Formato de respuesta (igual que OpenAI):')
print('# {"choices": [{"message": {"content": "..."}}], "usage": {...}}')

---
## PARTE 7 · Benchmark: rendimiento del servidor
**⏱ 5 minutos**

Una ventaja de vLLM sobre Ollama es el throughput. Midamos cuántos tokens por segundo genera.

In [ ]:
import time

# Medir throughput: tokens generados por segundo
print('=== BENCHMARK: tokens/segundo ===')

inicio = time.time()
resp_bench = client_local.chat.completions.create(
    model=MODELO_HF,
    messages=[{'role': 'user',
               'content': 'Escribe un párrafo largo sobre inteligencia artificial.'}],
    max_tokens=300
)
duracion = time.time() - inicio

tokens_generados = resp_bench.usage.completion_tokens
throughput = tokens_generados / duracion

print(f'Tokens generados   : {tokens_generados}')
print(f'Tiempo total       : {duracion:.2f}s')
print(f'Throughput         : {throughput:.1f} tokens/segundo')
print(f'\nPara comparación, GPT-4 produce ~40 tokens/s en condiciones normales.')
print('✅ Benchmark completado')

---
## 💬 Preguntas de reflexión

> **1. ¿Por qué vLLM usa `float16` en lugar de `float32` por defecto en la T4?**
> ¿Qué se sacrifica y qué se gana?
>
> *(Escribe aquí)*

> **2. Comparaste throughput con GPT-4. ¿Qué ventajas tiene usar tu propio servidor aunque sea más lento?**
> Piensa en privacidad, costo, personalización.
>
> *(Escribe aquí)*

> **3. vLLM y Ollama exponen la misma API de OpenAI. ¿Podrías cambiar tu app de Ollama a vLLM sin cambiar el código del cliente?**
>
> *(Escribe aquí)*

---

## ✅ Resumen

| Endpoint | URL |
|---|---|
| Chat completions | `TU_URL/v1/chat/completions` |
| Completions | `TU_URL/v1/completions` |
| Lista de modelos | `TU_URL/v1/models` |
| Health check | `TU_URL/health` |

**Guarda tu URL** — la necesitarás en las demás actividades del curso.

---
## 🧹 Limpieza (ejecutar al terminar la sesión)

In [ ]:
# Cerrar en orden: túnel Ngrok primero, luego el servidor vLLM
ngrok.kill()                    # Cierra el túnel
vllm_process.terminate()        # Detiene el proceso vLLM
vllm_process.wait()             # Espera a que termine limpiamente
print('✅ Servidor vLLM y túnel cerrados correctamente')